# 5.4 各參數能用不同步幅嗎？


兩格參數的梯度是1與100，都用學習率0.001，基本梯度下降會分別移0.001與0.1。Adam再參考每格「過去通常多大」的梯度，用大小歷史調整方向的尺度。

m保存梯度加權平均，v保存平方梯度加權平均，β1、β2各控制一份歷史的保留比例：新m＝β1×舊m＋(1−β1)×g，新v＝β2×舊v＋(1−β2)×g²。兩格各保存自己的歷史；下面取β1=0.9、β2=0.999，只追第一步。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch

g = torch.tensor([1.0, 100.0])
m = torch.zeros(2)
v = torch.zeros(2)
m = 0.9 * m + 0.1 * g
v = 0.999 * v + 0.001 * g.square()
m_hat = m / (1 - 0.9)
v_hat = v / (1 - 0.999)
update = 0.001 * m_hat / (v_hat.sqrt() + 1e-8)
print("方向平均", m, "平方平均", v)
print("原梯度", g, "應減去的更新量", update)

方向平均 tensor([ 0.1000, 10.0000]) 平方平均 tensor([1.0000e-03, 1.0000e+01])
原梯度 tensor([  1., 100.]) 應減去的更新量 tensor([0.0010, 0.0010])


輸入g為 `[1,100]`，歷史全0。第一步m得到 `[0.1,10]`，v得到 `[0.001,10]`。如果不修正，零歷史會讓早期平均偏小，所以第一步把m除以0.1、v除以0.001，得到 `m_hat=[1,100]`、`v_hat=[1,10000]`。這叫偏差修正；到第t步的分母分別是 `1-β1**t` 與 `1-β2**t`，t表示已經更新幾次。

最後用m_hat除以v_hat的平方根，再乘學習率0.001。兩格都約為0.001，而不是一格0.001、一格0.1；`1e-8` 是一億分之一的小正數，避免分母為0。輸出說「應減去」，因為這段只算移動量，沒有建立真正參數去執行更新。

第一步分子與分母同時來自同一梯度，兩格更新量近似；後續m、v記憶長度不同，不必再相同。到第t步，偏差修正分母是1−β1的t次方與1−β2的t次方。因此共同學習率不是每格實際位移，續訓也需保留m、v與步數。

練習只把第二個梯度100改成-100，先預測平方平均不變，方向平均第二格變負，應減去的更新量約-0.001，再執行核對。在這個從零歷史開始的第一步，應減去的更新量與當前梯度同號；減去負值會讓那格參數增加，方向含義與 [1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html) 一致。後續更新量的正負由歷史平均 `m_hat` 決定，可能與當前梯度不同號。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
